# 10 · Deblurring — blur dataset and classic baselines (Wiener, Richardson–Lucy)

**Where this sits on GSTP:** camera → FPGA (dark-reference subtraction, FIR filtering) → **deblurring (Jetson)** →
hazard models → ground.

**Why we blur on purpose.** There are no blurry GSTP images with a sharp "right answer" yet. So we take sharp
satellite images, blur them ourselves, and score every method against the sharp original.

**Blur types:** motion (platform motion / jitter → streaks), defocus (lens out of focus), Gaussian (general optical
softness), plus a *no extra blur* check group — a method should not damage an image that is already sharp.
Noise removal is the FPGA's job, so only a small leftover noise is added (placeholder until the FPGA team gives the
real level).

**Sharp images:** the same chips and splits as the hazard models — HLS Burn Scars (6 bands, 30 m) and Sen1Floods11
Sentinel-2 (12 bands, 10 m) — so deblurred test images can go through the burn-scar and flood models (notebook 11).

**Classic methods** need the blur shape (PSF). They are scored with the **true PSF** (best case — on orbit it is never
known this exactly) and with **one fixed "nominal" PSF** (what a lab measurement would give). Settings are tuned on
validation; the test set is scored once. Runtime on a T4: about 30–40 min.

In [ ]:
# ── Setup: run once per Colab session (Runtime ▸ Change runtime type ▸ T4 GPU) ──
import os, sys, subprocess, shutil
from pathlib import Path

REPO_URL = "https://github.com/inayatarshad/trainedmodels_gstp.git"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = Path("/content")
    DRIVE = Path("/content/drive/MyDrive/gstp_runs")       # checkpoints + results persist here
    REPO = WORK / "trainedmodels_gstp"
    if not REPO.exists():
        if subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO)]).returncode != 0:
            # Private repo? Upload the repo zip to MyDrive/trainedmodels_gstp.zip and re-run.
            shutil.unpack_archive("/content/drive/MyDrive/trainedmodels_gstp.zip", WORK)
            REPO = next(WORK.glob("trainedmodels_gstp*"))
    os.chdir(REPO)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
else:  # local / Kaggle
    REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    os.chdir(REPO)
    WORK = DRIVE = REPO / "runs"

sys.path.insert(0, str(REPO / "src"))
DRIVE.mkdir(parents=True, exist_ok=True)

import torch
from gstp.utils import seed_everything, get_device
seed_everything(42)
print("Repo:", REPO, "| device:", get_device(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


In [ ]:
MODULE = 'deblur'

from gstp.data import burn_scars as BS, sen1floods11 as S
from gstp.data.segmentation import split_samples
from gstp.data.deblur import DeblurEvalSet, guess_scale
from gstp import blur as B, deblur_classic as D
from gstp.utils import save_json, load_json, get_device
import numpy as np, pandas as pd, torch, tarfile, json, matplotlib.pyplot as plt

DEV = get_device()
BURN, FLOOD = WORK / "data" / "hls_burn_scars", WORK / "data" / "sen1floods11"
RESULTS = REPO / "results" / "deblur"
FIG = RESULTS / "figures"
CKPT_DIR = DRIVE / "deblur"
for p in (BURN, FLOOD, FIG, CKPT_DIR):
    p.mkdir(parents=True, exist_ok=True)

from huggingface_hub import snapshot_download
if not any(BURN.rglob("*merged*.tif")):
    snapshot_download("ibm-nasa-geospatial/hls_burn_scars", repo_type="dataset", local_dir=str(BURN))
    for t in BURN.rglob("*.tar.gz"):
        with tarfile.open(t) as tf:
            tf.extractall(BURN)
if not any(FLOOD.rglob("*_LabelHand.tif")):
    for sub in ("S2Hand", "LabelHand"):
        subprocess.run(["gsutil", "-m", "-q", "cp", "-r",
                        f"gs://sen1floods11/v1.1/data/flood_events/HandLabeled/{sub}", str(FLOOD)], check=True)
    subprocess.run(["gsutil", "-m", "-q", "cp", "-r", "gs://sen1floods11/v1.1/splits", str(FLOOD)], check=True)
    (FLOOD / "S1Hand").mkdir(exist_ok=True)   # reader looks for the folder; S1 radar is not used here

# Same splits as the hazard models (notebooks 01 and 02)
burn_train_all = BS.build_samples(BURN, "training")
burn = dict(zip(("train", "val"), split_samples(burn_train_all, 0.15, seed=42, group_key=lambda s: s.meta["tile"])))
burn["test"] = BS.build_samples(BURN, "validation")
flood = {sp: S.build_samples(FLOOD, sp, "s2") for sp in ("train", "val", "test")}
for sp in flood:   # S2-only: drop the S1 source the reader adds for other modalities
    for s in flood[sp]:
        s.sources = [src for src in s.sources if "S2Hand" in src[0]]
scale = {"burn": guess_scale(burn["train"], BS.NODATA), "flood": guess_scale(flood["train"])}
print("chips  burn:", {k: len(v) for k, v in burn.items()}, " flood:", {k: len(v) for k, v in flood.items()},
      " reflectance scale:", scale)

N_VAL = 30   # validation chips per dataset used for tuning / model selection
val_sets = [DeblurEvalSet(burn["val"][:N_VAL], scale["burn"], BS.NODATA, seed=101, name="HLS burn scars"),
            DeblurEvalSet(flood["val"][:N_VAL], scale["flood"], seed=102, name="Sen1Floods11 S2")]
test_sets = [DeblurEvalSet(burn["test"], scale["burn"], BS.NODATA, seed=201, name="HLS burn scars"),
             DeblurEvalSet(flood["test"], scale["flood"], seed=202, name="Sen1Floods11 S2")]
TYPE_NAMES = {"motion": "Motion", "defocus": "Defocus", "gaussian": "Gaussian", "none": "No extra blur (check)"}
RGB = {"HLS burn scars": (2, 1, 0), "Sen1Floods11 S2": (S.S2_BANDS.index("B4"), S.S2_BANDS.index("B3"), S.S2_BANDS.index("B2"))}

def pick(ds, t, n=2):
    """Index of an example chip of blur type t (the n-th one, so examples are not hand-picked)."""
    ks = [k for k, r in enumerate(ds.recipes) if r["blur_type"] == t]
    return ks[min(n, len(ks) - 1)]

def show(img, ds, crop=(slice(160, 352), slice(160, 352)), ref=None):
    """True-colour crop, stretched with the SHARP image's percentiles so all panels share one stretch."""
    x = img[list(RGB[ds])][:, crop[0], crop[1]].transpose(1, 2, 0)
    r = x if ref is None else ref[list(RGB[ds])][:, crop[0], crop[1]].transpose(1, 2, 0)
    lo, hi = np.percentile(r, 1), np.percentile(r, 99)
    return np.clip((x - lo) / max(hi - lo, 1e-4), 0, 1)

## 1. The blur types and the sample pairs (Day 1)

In [ ]:
rng = np.random.default_rng(7)
recipes = [B.sample_recipe(rng, t) for t in ("motion", "motion", "defocus", "gaussian")]
recipes[0].update(kind="line", length=11.0, angle=30.0); recipes[1].update(kind="jitter", length=12.0)
ex = DeblurEvalSet(flood["val"][3:4], scale["flood"], name="Sen1Floods11 S2", recipes=[recipes[0]])[0]
labels = ["Motion (straight line)", "Motion (jitter)", "Defocus", "Gaussian"]
fig, ax = plt.subplots(2, 5, figsize=(19, 8))
ax[1, 0].imshow(show(ex["sharp"], ex["dataset"])); ax[1, 0].set_title("Sharp original", fontsize=13)
ax[0, 0].text(0.5, 0.5, "Blur shape (PSF)\nzoomed in\n\n↓\n\nwhat it does\nto the image", ha="center", va="center",
              fontsize=13); 
for j, (r, lab) in enumerate(zip(recipes, labels), 1):
    y, psf = B.degrade(ex["sharp"], r)
    ax[0, j].imshow(psf, cmap="magma", interpolation="nearest"); ax[0, j].set_title(f"{lab}\nPSF {psf.shape[0]}×{psf.shape[1]} px", fontsize=12)
    ax[1, j].imshow(show(y.numpy(), ex["dataset"], ref=ex["sharp"])); ax[1, j].set_title(lab, fontsize=13)
for a in ax.ravel(): a.axis("off")
fig.suptitle("Blur types used to make the training and test images (Sentinel-2, 10 m, true colour)", fontsize=15)
plt.tight_layout(); fig.savefig(FIG / "D1_1_blur_types.png", dpi=130, bbox_inches="tight"); plt.show()

In [ ]:
# Sample pairs from the fixed test set: sharp original vs blurred input, both datasets, every blur type
fig, ax = plt.subplots(2, 8, figsize=(26, 7.5))
for row, ds in enumerate(test_sets):
    for j, t in enumerate(B.ALL_TYPES):
        i = pick(ds, t)
        it = ds[i]
        ax[row, 2 * j].imshow(show(it["sharp"], ds.name)); ax[row, 2 * j].set_title(f"{ds.name}\nsharp", fontsize=11)
        ax[row, 2 * j + 1].imshow(show(it["blurred"], ds.name, ref=it["sharp"]))
        ax[row, 2 * j + 1].set_title(f"blurred: {TYPE_NAMES[t]}", fontsize=11)
for a in ax.ravel(): a.axis("off")
fig.suptitle("Sample pairs: sharp original (the right answer) | blurred input (what the methods get)", fontsize=15)
plt.tight_layout(); fig.savefig(FIG / "D1_2_sample_pairs.png", dpi=110, bbox_inches="tight"); plt.show()

split = pd.DataFrame({"Dataset": ["HLS burn scars (6 bands, 30 m)", "Sen1Floods11 Sentinel-2 (12 bands, 10 m)"],
                      "Train chips": [len(burn["train"]), len(flood["train"])],
                      "Validation chips": [len(burn["val"]), len(flood["val"])],
                      "  (used for tuning)": [len(val_sets[0]), len(val_sets[1])],
                      "Test chips": [len(burn["test"]), len(flood["test"])]})
split.to_csv(RESULTS / "dataset_split.csv", index=False)
save_json({ds.name: ds.recipe_table() for ds in test_sets}, RESULTS / "test_degradations.json")
print("test chips per blur type:", pd.Series([r["blur_type"] for ds in test_sets for r in ds.recipes]).value_counts().to_dict())
split

## 2. Tune the classic methods on validation (Day 2)

In [ ]:
val_items = [it for ds in val_sets for it in ds]
best = D.tune(val_items, device=DEV)
save_json(best, RESULTS / "classic_tuned.json"); save_json(best, CKPT_DIR / "classic_tuned.json")
fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
for mode, ls in (("true", "o-"), ("nominal", "s--")):
    c = best[f"wiener_{mode}"]["curve"]; ax[0].semilogx(list(c), list(c.values()), ls, label=f"{mode} PSF")
    c = best[f"rl_{mode}"]["curve"]; ax[1].plot(list(c), list(c.values()), ls, label=f"{mode} PSF")
base = np.mean([D.score(i["sharp"], i["blurred"], i["valid"], ssim=False)["psnr"] for i in val_items])
for a, t in zip(ax, ("Wiener filter: noise constant K", "Richardson–Lucy: number of iterations")):
    a.axhline(base, color="grey", ls=":", label="no deblurring"); a.set_title(t); a.set_ylabel("validation PSNR (dB)")
    a.grid(alpha=.3); a.legend()
plt.tight_layout(); fig.savefig(FIG / "D2_1_classic_tuning.png", dpi=130); plt.show()
{k: v["param"] for k, v in best.items()}

## 3. Score the test set once

In [ ]:
methods = D.method_table(best)
keep = {ds.samples[pick(ds, t)].id for ds in test_sets for t in B.ALL_TYPES}
rows, kept = [], {}
for ds in test_sets:
    r, k = D.evaluate(ds, methods, device=DEV, keep=keep)
    rows += r; kept.update(k)
df = pd.DataFrame(rows)
df.to_csv(RESULTS / "test_classic_per_image.csv", index=False)
order = list(methods)
overall = df.groupby("method")[["psnr", "ssim"]].mean().loc[order]
by_type = df.pivot_table(index="method", columns="blur_type", values="psnr").loc[order, list(B.ALL_TYPES)]
ssim_type = df.pivot_table(index="method", columns="blur_type", values="ssim").loc[order, list(B.ALL_TYPES)]
save_json({"overall": overall.to_dict("index"), "psnr_by_blur_type": by_type.to_dict("index"),
           "ssim_by_blur_type": ssim_type.to_dict("index"), "tuned": {k: v["param"] for k, v in best.items()}},
          RESULTS / "test_classic.json")
display(overall.style.format({"psnr": "{:.2f} dB", "ssim": "{:.4f}"}))
display(by_type.rename(columns=TYPE_NAMES).style.format("{:.2f}").set_caption("Test PSNR (dB) by blur type"))
display(ssim_type.rename(columns=TYPE_NAMES).style.format("{:.4f}").set_caption("Test SSIM by blur type"))

In [ ]:
# Side by side: blurry | Wiener | Richardson-Lucy | sharp, one row per blur type (flood chips, true PSF)
items = {it["id"]: it for it in test_sets[1] if it["id"] in keep}
cols = [("No deblurring", "Blurry input")] + [(m, m.split(" (")[0]) for m in order if "true" in m]
fig, ax = plt.subplots(4, len(cols) + 1, figsize=(4.2 * (len(cols) + 1), 17))
for i, t in enumerate(B.ALL_TYPES):
    cid = next(c for c, it in items.items() if it["blur_type"] == t); it = items[cid]
    for j, (m, title) in enumerate(cols):
        p = df[(df.id == cid) & (df.method == m)].psnr.iloc[0]
        ax[i, j].imshow(show(kept[cid][m], it["dataset"], ref=it["sharp"])); ax[i, j].set_title(f"{title}\n{p:.1f} dB", fontsize=12)
    ax[i, -1].imshow(show(it["sharp"], it["dataset"])); ax[i, -1].set_title("Sharp original", fontsize=12)
    ax[i, 0].set_ylabel(TYPE_NAMES[t], fontsize=14); ax[i, 0].set_xticks([]); ax[i, 0].set_yticks([])
for a in ax[:, 1:].ravel(): a.axis("off")
for a in ax[:, 0]: [s.set_visible(False) for s in a.spines.values()]
fig.suptitle("Classic deblurring with the true blur shape — test images (PSNR vs sharp original, higher is better)", fontsize=15)
plt.tight_layout(); fig.savefig(FIG / "D2_2_classic_side_by_side.png", dpi=100, bbox_inches="tight"); plt.show()

fig, ax = plt.subplots(figsize=(12, 4.5))
by_type.rename(columns=TYPE_NAMES).T.plot.bar(ax=ax, rot=0, width=0.8)
ax.set_ylabel("test PSNR (dB)"); ax.set_ylim(by_type.values.min() - 2, by_type.values.max() + 1.5); ax.grid(axis="y", alpha=.3)
ax.set_title("Classic methods by blur type (test set)"); ax.legend(fontsize=9, ncol=2)
plt.tight_layout(); fig.savefig(FIG / "D2_3_classic_psnr_by_type.png", dpi=130); plt.show()

In [ ]:
# ── Package results: metrics + figures go to Drive as a zip you can download ──
import shutil
archive = shutil.make_archive(str(DRIVE / f"results_{MODULE}"), "zip", RESULTS)
print("Results zip :", archive)
print("Checkpoints :", sorted(str(p) for p in CKPT_DIR.glob("*.pt")))